# Simple DP2 Injection Demo

Load one DP2 patch, then run three examples:

1. Inject 100 clusters in every band in `BANDS`.
2. Inject 100 clusters in `SELECTED_BAND`.
3. Run 10 rounds of 100 clusters in every band: 1,000 attempted injections per band.

Use your working CANFAR Rubin kernel and an up-to-date INJECT installation. Edit the data ID and bands below to match your data. Each round starts from the original image, not the previously injected image. Multiband runs use the same positions and apparent magnitudes, but a separate Rubin image and PSF in each band. This is a workflow demo, not a physical colour model or calibrated completeness test.

## 1. Imports
Run this in the Rubin kernel where INJECT is installed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler
from inject import ClusterConfig, InjectionConfig, InjectionPipeline, notebook_output_dir

## 2. Settings
Change these values. The data ID below comes from the earlier DP2 example; each chosen band must exist for that patch. DP2 calibrated nJy images use an AB zero point of about 31.4 and a pixel scale of about 0.2 arcsec/pixel. Verify these if using another product. Radii below are in pixels.

In [ ]:
BUTLER_REPO = 'https://ws-uv.canfar.net/lsst/api/butler/repo/dp2/butler.yaml'
BUTLER_COLLECTIONS = 'dp2'
DATA_ID = {'skymap': 'lsst_cells_v2', 'tract': 7508, 'patch': 0}
BANDS = ['g', 'r', 'i']
SELECTED_BAND = 'r'
N_CLUSTERS = 100
N_ROUNDS = 10
OUTPUT_DIR = notebook_output_dir('simple_dp2_pipeline_demo')

config = InjectionConfig(
    bands=BANDS, band=SELECTED_BAND, n_clusters=N_CLUSTERS,
    cutout_size=800, edge_buffer=60, seed=42,
    use_actual_psf=True, add_noise=False, pixel_scale=0.2, zero_point=31.4,
    output_dir=str(OUTPUT_DIR / 'selected'),
    cluster_config=ClusterConfig(
        profile_type='king', mag_min=20.5, mag_max=24.5,
        r_half_min=2.0, r_half_max=6.0,
        concentration_min=5.0, concentration_max=20.0,
    ),
)

## 3. Load the data
Use the CANFAR session credentials. Load the bands once and reuse them below. The package handles DP2 CellCoadd conversion; no conversion code is needed here. Full coadds are fetched before cropping, so loading may take a while.

In [ ]:
butler = Butler(BUTLER_REPO, collections=BUTLER_COLLECTIONS)
pipe = InjectionPipeline(config)
pipe.load_data(butler=butler, data_id=DATA_ID, dataset_type='deep_coadd')

print({band: image.shape for band, image in pipe.images.items()})

## 4. Simple MCI detection
This detector uses matched filtering and MCI. Change its settings below or replace it with your own function later. `DETECTOR_PSF_FWHM` is a simple shared detection setting; injection still uses each band's actual local Rubin PSF.

Recovery uses the nearest detection within `MATCH_RADIUS` pixels. Matches are not one-to-one, and pre-existing sources are not removed, so treat the fractions as demo results rather than science completeness.

In [ ]:
import pandas as pd
from inject import ClusterRetrieval
from inject.detection import run_cluster_detection

MATCH_RADIUS = 5.0
DETECTOR_PSF_FWHM = 3.5

def detector(image):
    return run_cluster_detection(
        image, psf_fwhm=DETECTOR_PSF_FWHM, pixel_scale=config.pixel_scale,
        threshold_sigma=3.0, snr_min=3.0, mci_max=0.85,
        r_half_min=1.0, ellipticity_max=0.6, box_size=64,
        use_multiscale=True, use_mci=True, verbose=False,
    )

## 5. Inject and recover 100 clusters in all listed bands
One shared catalog, one original image and Rubin PSF per band. Run MCI on each injected image and match that band's detections to its truth catalog. Save the recovery table and show the injected images.

In [ ]:
all_bands = pipe.run_batch_multiband(
    n_iterations=1, n_per_iter=N_CLUSTERS, detector_fn=detector,
    checkpoint_dir=str(OUTPUT_DIR / 'all_bands'),
    store_images=True, use_psf_cache=False, n_workers=1, verbose=False,
)

all_recovery = []
fig, axes = plt.subplots(1, len(BANDS), figsize=(5 * len(BANDS), 5), constrained_layout=True)
for axis, band in zip(np.atleast_1d(axes), BANDS):
    result = all_bands[band][0]
    rows = result['injection_info']
    assert rows and all(row['psf_used'] == 'rubin' for row in rows), 'Check Rubin PSF evaluation.'
    retrieval = ClusterRetrieval(rows, result['detections'])
    retrieval.match_detections(match_radius=MATCH_RADIUS)
    all_recovery.append({'band': band, **retrieval.get_summary_statistics()})
    image = result['injected_image']
    vmin, vmax = np.nanpercentile(pipe.images[band], [1, 99])
    axis.imshow(image, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    axis.set_title(f'{band}: {len(rows)} injected')
    axis.set_axis_off()
plt.show()
all_recovery = pd.DataFrame(all_recovery)
all_recovery.to_csv(OUTPUT_DIR / 'all_bands_recovery.csv', index=False)
all_recovery[['band', 'n_injected', 'n_detected', 'overall_completeness']]

## 6. Inject and recover 100 clusters in one selected band
Use a clean original image for `SELECTED_BAND`. Call `detect_with()` for MCI and `analyze()` for recovery. Save the catalogs and show original, injected, and difference images.

In [ ]:
pipe.image = pipe.images[SELECTED_BAND]
injected_image, injection_info = pipe.inject(band=SELECTED_BAND, verbose=False)
assert injection_info and all(row['psf_used'] == 'rubin' for row in injection_info), 'Check Rubin PSF evaluation.'
pipe.detect_with(detector)
selected_recovery = pipe.analyze(match_radius=MATCH_RADIUS)
pipe.save_results()

difference = injected_image.astype(float) - pipe.image.astype(float)
vmin, vmax = np.nanpercentile(pipe.image, [1, 99])
limit = max(float(np.nanmax(np.abs(difference))), 1e-6)
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for axis, image, title in zip(axes[:2], [pipe.image, injected_image], ['Original', 'Injected']):
    axis.imshow(image, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    axis.set_title(f'{SELECTED_BAND}: {title}')
axes[2].imshow(difference, origin='lower', cmap='RdBu_r', vmin=-limit, vmax=limit)
axes[2].set_title('Difference')
for axis in axes:
    axis.set_axis_off()
plt.show()
pd.DataFrame([{'band': SELECTED_BAND, **selected_recovery}])[['band', 'n_injected', 'n_detected', 'overall_completeness']]

## 7. Batch detection and recovery: 10 rounds of 100 per band
Each round uses a clean original image. Run MCI and match separately within each band and round; never cross-match pooled detections from different images. Add the matched counts afterwards for a total recovery fraction per band.

Per-round catalogs are saved under `batch/<band>`. The recovery CSV has one row per band and round. The current multiband helper does not forward per-band masks, so this is not a mask-screened science run.

In [ ]:
batch = pipe.run_batch_multiband(
    n_iterations=N_ROUNDS, n_per_iter=N_CLUSTERS, detector_fn=detector,
    checkpoint_dir=str(OUTPUT_DIR / 'batch'),
    store_images=False, use_psf_cache=False, n_workers=1, verbose=False,
)

batch_recovery = []
for band, rounds in batch.items():
    for result in rounds:
        rows = result['injection_info']
        assert rows and all(row['psf_used'] == 'rubin' for row in rows), 'Check Rubin PSF evaluation.'
        retrieval = ClusterRetrieval(rows, result['detections'])
        retrieval.match_detections(match_radius=MATCH_RADIUS)
        batch_recovery.append({
            'band': band, 'round': result['iteration'] + 1, **retrieval.get_summary_statistics(),
        })
batch_recovery = pd.DataFrame(batch_recovery)
batch_recovery.to_csv(OUTPUT_DIR / 'batch_recovery.csv', index=False)
batch_totals = batch_recovery.groupby('band')[['n_injected', 'n_detected']].sum()
batch_totals['overall_completeness'] = batch_totals['n_detected'] / batch_totals['n_injected']
print('Outputs:', OUTPUT_DIR)
batch_totals